# Parametric angle-PINN — one network for a box of initial conditions

A single network $N(\phi,\theta)\to(u,p_r,L)$ learns the 2PN orbits for every $\theta=(p_{0}, r_0, p_{r0}, \nu)$ in a 4-parameter box. All functions live in `gravinns.parametric`; this notebook only drives the workflow:

1. **Training** (Table VI) — anchors from the reference database, then `train_parametric_pinn`.
2. **Test database** — integrate the test orbits once (`build_reference_cache`).
3. **Evaluation** (Tables IV and V) — score a trained model on the test orbits, heatmaps, eccentricity bins.
4. **Statistics for the paper** (Tables IV, V, VI) — paired PINN vs supervised comparisons.

**Files.** The six trained models are in `models/parametric/`. The anchor database `refs_20000_10orb.npz` and the test databases go in `data/` (see `data/README.md`; they are not all stored on GitHub).

## 0. Setup

Paths are relative to the repository root, so the notebook works wherever you start Jupyter (as long as it is inside the repository). Change `RUNS_DIR` to send new training runs elsewhere.

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

# repository root = first parent folder that contains pyproject.toml
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA_DIR   = str(REPO / "data")                    # reference / test orbit databases (.npz)
MODELS_DIR = str(REPO / "models" / "parametric")   # the six trained models of the paper
RUNS_DIR   = str(REPO / "runs")                    # where NEW training runs are written

import gravinns.constants as const
from gravinns.utils import get_device, set_global_seed
from gravinns.parametric import *

set_global_seed(42)
device = get_device(verbose=True)
const.print_normalization_summary()
const.check_relativistic_safety()
print("models available:", sorted(os.listdir(MODELS_DIR)))

## 1. Training of the models — the parametric surrogate (Table VI)

Table VI was produced with `n_anchors` = 500, 5000 and 20000, and two training modes:

* **fully supervised**: `pretrain_epochs=1_500_000`
* **PINN**: `pretrain_epochs=40000`, `curriculum_epochs=50000`, `pde_epochs=1410000`

`refs_20000_10orb.npz` was used as the anchor orbits for every trained model in the paper. Put it in `data/` before running this section.

### 1.1 Load the anchor database and choose the anchors

In [ ]:
# 1. Load the big database
cache = load_reference_cache(os.path.join(DATA_DIR, "refs_20000_10orb.npz"))

In [ ]:
# 2. Choose the number of anchors (the first n_anchors orbits of the database)
n_anchors = 2000          # paper: 500, 5000 or 20000
anchor_indices = range(n_anchors)

# 3. Define common phi grid
phi_max_global = min(cache['span'][anchor_indices])

# 4. Build ref_data: every anchor orbit on the common phi grid
ref_data = build_ref_data_from_cache(cache, anchor_indices, phi_max_global, n_pts=2000)

# 5. Extract anchor thetas for probes
anchor_thetas = [tuple(row) for row in cache['thetas'][anchor_indices]]

# 6. Manually create probes (3 seen + 3 unseen)
seen_thetas = [anchor_thetas[i] for i in range(3)]
unseen_thetas = [tuple(cache['thetas'][n_anchors + i]) for i in range(3)]   # next 3 orbits
probe_thetas = seen_thetas + unseen_thetas
probe_is_seen = [True]*3 + [False]*3
print(f"{n_anchors} anchors, phi_max_global = {phi_max_global:.4f} rad")

### 1.2 Training

Every setting of the run is below; this is the configuration of the PINN mode in the paper. For the fully supervised mode set `pretrain_epochs=1_500_000`.

Note: `make_probe_thetas(...)` below replaces the 3 + 3 probes of step 6 with 2 + 2 probes; delete that call to keep the probes of step 6.

After training, the orbit of any in-range $\theta$ is `predict_orbit(model, p0_factor, r0_km, pr0_factor, nu)`.

In [ ]:
# ── USAGE: parametric PINN over a 4-parameter box ────────────────────────
# One network learns orbits across a RANGE of (p0_factor, r0_km, pr0_factor,
# nu). Supervised warm-start on a few 2PN anchor orbits, then a curriculum
# that blends in the PDE, then pure PDE at the chosen PN order over the whole
# box. After training, query any UNSEEN theta in-range for its orbit.

param_ranges = dict(
    p0_factor  = (0.6, 1.2),
    r0_km      = (100.0, 400.0),
    pr0_factor = (-0.35, 0.35),
    nu         = (0.001, 0.25),
)


# ── Anchor thetas are now generated AUTOMATICALLY from param_ranges. ──────
# parameter so a small set still spans the whole box (uniform-random clumps).
# Every candidate is validated: unbound orbits (E>=0) and ecc>max_ecc are
# rejected and resampled, so all anchors are physically usable.
#
#   n_anchors        how many supervised orbits (each costs one RK integration)
#   include_corners  also pin the 2^4=16 box corners (worst extrapolation
#                    spots). Only worth it for n_anchors >= 20.
#   seed             reproducibility

# To hand-pick instead, just assign a list of (p0_factor, r0_km, pr0_factor, nu):
# anchor_thetas = [(0.48, 110.0, 0.25, 0.24), (0.55, 120.0, 0.30, 0.22), ...]

# ── Probe orbits shown live: 2 TRAINING anchors + 2 UNSEEN held-out thetas.
# The anchors show whether the fit is holding; the unseen thetas show whether
# the network actually GENERALISES across the box -- which is the entire point
# of a parametric PINN. Each panel is titled SEEN/UNSEEN with its live L2.
# (Omit probe_thetas entirely and the trainer picks 2+2 for you.)

probe_thetas, probe_is_seen = make_probe_thetas(
    param_ranges, anchor_thetas,
    n_seen   = 2,      # how many training anchors to show
    n_unseen = 2,      # how many held-out thetas to show
    min_dist = 0.01,   # unseen probes must be this far (normalised) from anchors
    seed     = 1,
)



model, hist, heat_hist = train_parametric_pinn(
    param_ranges,
    anchor_thetas,
    probe_thetas   = probe_thetas,
    probe_is_seen  = probe_is_seen,
    ref_data=ref_data,          # <-- PASS THE PRECOMPUTED DATA
    heatmap_axes   = ("p0_factor", "r0_km"),  # the 2D error map varies these two
    pde_pn_order   = 2,        # physics phase PN order: 0 / 1 / 2 / 3 (default 2)
    data_pn_order  = 2,        # supervised orbits' PN order
    n_orbits       = 10,        # 3-5
    n_pts          = 4000,
    pretrain_epochs   = 40000, # Phase 1: supervised MSE. 30k drove l_sup
                               # to 1e-11 -- pointless overfit; capacity
                               # floor is ~1e-4. Budget -> PDE phase.
    curriculum_epochs = 50000, # Phase 2: blend supervised -> PDE
    pde_epochs        = 1410000, # Phase 3: pure PDE over the box
    w_sup=1.0,
    w_energy=0.0,          # OFF: H=E* is already implied by the Hamilton eqs
    n_theta_batch=64,      # only used when use_causal=True
    n_colloc=8000,         # gradient noise halves at 8000 vs 1500
    n_sup_batch=16000,      # supervised minibatch: 8000 random (anchor,point)
                           #   pairs per epoch -- epoch cost stays flat no
                           #   matter how many anchors you add.
    # alpha_floor is left at its None default: with data_pn_order ==
    # pde_pn_order it resolves to 1.0 (FULL supervision all the way -- the
    # anchors are exact solutions of the enforced PDE, so there is nothing to
    # release, and they are what pins the phase at unseen theta). For a
    # transfer run (data at a lower order) it resolves to 0.05 as before.
    n_fourier=10, fourier_sigma=3.0,
    n_fourier_hi=16, fourier_sigma_hi=8.0,
    n_harmonic=8, hidden=384, depth=6, omega_hidden=64,
    lr=1e-3,               # cosine-annealed to lr*1e-3 automatically
    use_causal=False,      # iid (phi,theta) Monte-Carlo, as the working code
    heatmap_side=11,
    plot_every=5000, log_every=1000,
    base_dir=os.path.join(RUNS_DIR, "parametric_pinn_1_5mill_2000anchor_2"),   # output folder
    device=device)


## 2. Test orbits database

Run this to generate the test orbits (or training anchor orbits). `batch_size` can be increased depending on GPU VRAM. With `n_samples = 100000` the file is about 3 GB, so it is **not** stored on GitHub: generate it once with this cell. It uses `anchor_thetas` from section 1.1 to compute each test orbit's distance to the nearest anchor.

In [ ]:
# ── integrate the 2000 orbits ONCE, score any number of models ──
# build_reference_cache stores the RK reference orbits themselves, so every
# later model (Adam-phase, refined, retrained, different capacity) is scored
# by pure network inference -- no integration, seconds per model -- on the
# IDENTICAL orbits. That makes the comparison PAIRED: "improved on X% of the
# same 2000 orbits" and the median ratio, which are much stronger statistics
# than comparing two independent random samples.
#
# Verified: cache-scored L2 reproduces the integrate-inline value to ~7e-8
# relative (float32 storage), so nothing is lost by caching.
#
# n_orbits / n_pts / pde_pn_order define the cache; keep them fixed and reuse
# the same file for every model you want to compare.

# ═══ 1. BUILD ONCE (GPU-batched; ~7 MB at 2000 x 1500) ═══════════════════
cache = build_reference_cache(
    param_ranges,
    path          = os.path.join(DATA_DIR, "refs_10orb_test.npz"),
    n_samples     = 100000,
    anchor_thetas = anchor_thetas,   # enables the distance-
    pde_pn_order  = 2,
    n_orbits      = 10,               # ← match the model's training run
    n_pts         = 3000,
    n_steps       = 12000,
    seed          = 5678,            # same seed as build_benchmark_db -> same thetas
    batch_size    = 256,
    device        = device)


## 3. Evaluation of the models (Tables IV and V)

The models are evaluated on unseen test orbits, the same memory-safe way as section 4:

1. load the test database once (`load_reference_cache_lazy`, float32, one database in memory),
2. choose the model,
3. score it with `score_once` (the model is loaded, scored and freed; the per-orbit L2 is saved as `l2_<tag>.npy`), then build a light `db` for the heatmaps (`score_db`, no big arrays).

For a 100 000-orbit database, restart the kernel and run only the setup cell and this section.

In [ ]:
# 1. Load reference cache (contains eccentricities and true values) -- ONE cache at a time
cache = load_reference_cache_lazy(os.path.join(DATA_DIR, "refs_10orb_test.npz"))
L2_DIR = "."                             # where l2_<tag>.npy files are written

In [ ]:
# 2. Choose the model you want to evaluate (pick one)
#   PINN,       K=20000: "parametric_pinn_1_5mill_PINN_20000"
#   supervised, K=20000: "parametric_pinn_1_5mill_supervised_20000"
#   PINN,       K=5000 : "parametric_pinn_1_5mill_PINN_5000"
#   supervised, K=5000 : "parametric_pinn_1_5mill_supervised_5000"
#   PINN,       K=500  : "parametric_pinn_1_5mill_PINN_500"
#   supervised, K=500  : "parametric_pinn_1_5mill_supervised_500"
MODEL_NAME = "parametric_pinn_1_5mill_PINN_20000"
WHICH      = "param_pinn_best.pt"      # or "param_pinn.pt" (final) / "param_pinn_last.pt"
TAG        = "eval_" + MODEL_NAME.replace("parametric_pinn_1_5mill_", "")   # -> l2_eval_PINN_20000.npy

In [ ]:
# 3. Score the model on every test orbit (model loaded, scored, freed), then a light db
l2 = score_once(os.path.join(MODELS_DIR, MODEL_NAME), WHICH, TAG, cache,
                device=device, batch_size=16, out_dir=L2_DIR)
# already scored before? skip the line above and use:
# l2 = np.load(os.path.join(L2_DIR, f"l2_{TAG}.npy"))
db = score_db(cache, l2)

In [ ]:
# 4. Heatmaps
db_pair_heatmaps(db, nbins=12, stat="frac", tol=1e-3)   # fraction below tolerance
db_pair_heatmaps(db, nbins=12, stat="median")           # median L2 error per bin

In [ ]:
# 5. Eccentricity-bin table and boxplots (single model)
ecc_bins = [0.0, 0.3, 0.5, 0.65, 0.8]
tol = 1e-3   # same tolerance as used in the heatmaps

ecc_bin_table(cache["ecc"], db["l2"], ecc_bins=ecc_bins, tol=tol)
ecc_bin_boxplot(cache["ecc"], db["l2"], ecc_bins=ecc_bins)

## 4. Statistics for the paper (Tables IV, V and VI)

Paired comparison of the PINN and the supervised model at the same number of anchors K, on the SAME test orbits. One test database at a time; each model is scored, its per-orbit L2 saved as `l2_<tag>.npy`, and freed before the next one is loaded.

**Memory.** A 100 000-orbit database needs ~3.6 GB once loaded, and a second copy (e.g. the one section 3 keeps alive through `db`) is enough to crash the kernel. Safest: **restart the kernel, run the setup cell (section 0), then only this section.** If you don't restart, the next cell frees everything sections 1-3 left behind.

**Choose the test database** here (`refs_10orb_test.npz`, 100 000 test orbits).

In [ ]:
# Free what sections 1-3 left in memory (anchor database, ref_data, test database,
# the evaluated db -- which still references its database -- and the loaded model).
import gc
for _name in ("db", "model", "cache", "ref_data", "hist", "heat_hist",
              "anchor_thetas", "probe_thetas"):
    globals().pop(_name, None)
try:
    get_ipython().run_line_magic("reset", "-f out")   # IPython's Out[] history can also hold arrays
except Exception:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
memory_report("before section 4")

In [ ]:
TEST_DB = os.path.join(DATA_DIR, "refs_10orb_test.npz")

cache = load_reference_cache_lazy(TEST_DB)
ecc, th, rp, nu = cache_strata(cache)    # eccentricity, thetas, r_p / r_g, nu
L2_DIR = "."                             # where l2_<tag>.npy files are written

### Models PINN, K=20000 and supervised, K=20000

In [ ]:
l2_pinn20000 = score_once(os.path.join(MODELS_DIR, "parametric_pinn_1_5mill_PINN_20000"), "param_pinn_best.pt",
                        "pinn20000", cache, device=device, batch_size=16, out_dir=L2_DIR)
l2_sup20000  = score_once(os.path.join(MODELS_DIR, "parametric_pinn_1_5mill_supervised_20000"), "param_pinn_best.pt",
                        "sup20000",  cache, device=device, batch_size=16, out_dir=L2_DIR)

joint_table(l2_pinn20000, ecc, rp, label="PINN, K=20000")
joint_table(l2_sup20000,  ecc, rp, label="supervised, K=20000")
variance_decomposition(l2_pinn20000, ecc, rp, nu)
compare_models(l2_pinn20000, l2_sup20000, ecc, rp,
               name_a="PINN K=20000", name_b="supervised K=20000")

In [ ]:
# ONE joint grid: rows = eccentricity, columns = periapsis (fraction below 1e-3)
joint_table_2d(l2_pinn20000, ecc, rp, label="PINN, K=20000, n=100000")
joint_table_2d(l2_sup20000,  ecc, rp, label="supervised, K=20000, n=100000")
compare_models_joint(l2_pinn20000, l2_sup20000, ecc, rp,
                     name_a="PINN", name_b="supervised")

### Models PINN, K=500 and supervised, K=500

In [ ]:
l2_pinn500 = score_once(os.path.join(MODELS_DIR, "parametric_pinn_1_5mill_PINN_500"), "param_pinn_best.pt",
                        "pinn500", cache, device=device, batch_size=16, out_dir=L2_DIR)
l2_sup500  = score_once(os.path.join(MODELS_DIR, "parametric_pinn_1_5mill_supervised_500"), "param_pinn_best.pt",
                        "sup500",  cache, device=device, batch_size=16, out_dir=L2_DIR)

joint_table(l2_pinn500, ecc, rp, label="PINN, K=500")
joint_table(l2_sup500,  ecc, rp, label="supervised, K=500")
variance_decomposition(l2_pinn500, ecc, rp, nu)
compare_models(l2_pinn500, l2_sup500, ecc, rp,
               name_a="PINN K=500", name_b="supervised K=500")

In [ ]:
# ONE joint grid: rows = eccentricity, columns = periapsis (fraction below 1e-3)
joint_table_2d(l2_pinn500, ecc, rp, label="PINN, K=500, n=100000")
joint_table_2d(l2_sup500,  ecc, rp, label="supervised, K=500, n=100000")
compare_models_joint(l2_pinn500, l2_sup500, ecc, rp,
                     name_a="PINN", name_b="supervised")

### Models PINN, K=5000 and supervised, K=5000

In [ ]:
l2_pinn5000 = score_once(os.path.join(MODELS_DIR, "parametric_pinn_1_5mill_PINN_5000"), "param_pinn_best.pt",
                        "pinn5000", cache, device=device, batch_size=16, out_dir=L2_DIR)
l2_sup5000  = score_once(os.path.join(MODELS_DIR, "parametric_pinn_1_5mill_supervised_5000"), "param_pinn_best.pt",
                        "sup5000",  cache, device=device, batch_size=16, out_dir=L2_DIR)

joint_table(l2_pinn5000, ecc, rp, label="PINN, K=5000")
joint_table(l2_sup5000,  ecc, rp, label="supervised, K=5000")
variance_decomposition(l2_pinn5000, ecc, rp, nu)
compare_models(l2_pinn5000, l2_sup5000, ecc, rp,
               name_a="PINN K=5000", name_b="supervised K=5000")

In [ ]:
# ONE joint grid: rows = eccentricity, columns = periapsis (fraction below 1e-3)
joint_table_2d(l2_pinn5000, ecc, rp, label="PINN, K=5000, n=100000")
joint_table_2d(l2_sup5000,  ecc, rp, label="supervised, K=5000, n=100000")
compare_models_joint(l2_pinn5000, l2_sup5000, ecc, rp,
                     name_a="PINN", name_b="supervised")